
**Transform raw data into _cleaned,enriched_ data**.
1. Fix duplicates
2. Fix data types, missing & null values


In [0]:
# TODO: Merge/UPSERT, Study about DLT

In [0]:
from pyspark.sql.functions import col, current_timestamp, trim, lower, to_date, row_number, lit, when, round as s_round
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window


In [0]:
# DECLARE CONSTANTS
CATALOG = "retail_store"
BRONZE = "bronze"
SILVER = "silver"
BRONZE_SCHEMA_PATH = f"{CATALOG}.{BRONZE}"
SILVER_SCHEMA_PATH = f"{CATALOG}.{SILVER}"

# Product Inventory table

In [0]:
#Path to product_inventory broze table
BRONZE_TABLE_PATH = f"{BRONZE_SCHEMA_PATH}.product_inventory"

# Read bronze table
df_products_invt_bronze = spark.table(BRONZE_TABLE_PATH)

# 1. Deduplicate Bronze data 
df_bronze_clean = spark.table(f"{BRONZE_SCHEMA_PATH}.product_inventory").dropDuplicates(["ProductID"])

# 2. Transformed Silver DataFrame
df_silver_prd_inv = (
    df_bronze_clean
    .select(
        col("ProductID").cast(IntegerType()).alias("product_id"),
        col("ProductName").alias("product_name"),
        trim(lower(col("Category"))).alias("category"), 
        col("StockLevel").cast(IntegerType()).alias("stock_level"),
        col("Price").cast(DoubleType()).alias("price")
    )
    .filter(
        (col("product_id").isNotNull()) & 
        (col("price") >= 0) # Business Logic: Prices can't be negative
    )
    .withColumn("_processed_timestamp", current_timestamp())
)

# 3 check dataframe before wiritng to table

df_silver_prd_inv.show(5,truncate=False)


+----------+------------+---------------+-----------+-----+--------------------------+
|product_id|product_name|category       |stock_level|price|_processed_timestamp      |
+----------+------------+---------------+-----------+-----+--------------------------+
|1         |Product_1   |clothing       |22         |46.11|2026-03-16 10:33:48.711388|
|2         |Product_2   |home & kitchen |140        |81.6 |2026-03-16 10:33:48.711388|
|3         |Product_3   |home & kitchen |473        |78.72|2026-03-16 10:33:48.711388|
|4         |Product_4   |clothing       |386        |22.06|2026-03-16 10:33:48.711388|
|5         |Product_5   |beauty & health|284        |17.97|2026-03-16 10:33:48.711388|
+----------+------------+---------------+-----------+-----+--------------------------+
only showing top 5 rows


In [0]:
# 4. write to silver prduct inventory table
(df_silver_prd_inv.write
    .format("delta")
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .saveAsTable(f"{SILVER_SCHEMA_PATH}.product_inventory"))

print(f"✓ Silver Inventory created: {df_silver_prd_inv.count()} rows.")

✓ Silver Inventory created: 200 rows.


In [0]:
# 5. Read silver table

display(spark.table(f"{SILVER_SCHEMA_PATH}.product_inventory").limit(5))

product_id,product_name,category,stock_level,price,_processed_timestamp
56,Product_56,electronics,429,70.21,2026-03-16T10:33:50.392Z
73,Product_73,home & kitchen,188,68.55,2026-03-16T10:33:50.392Z
110,Product_110,beauty & health,364,25.42,2026-03-16T10:33:50.392Z
120,Product_120,electronics,354,30.43,2026-03-16T10:33:50.392Z
146,Product_146,beauty & health,388,11.09,2026-03-16T10:33:50.392Z


# Sales Transactions table

In [0]:
#Path to product_inventory broze table
BRONZE_SALES_TABLE_PATH = f"{BRONZE_SCHEMA_PATH}.sales_transactions"

# Read bronze table
df_sales_trx_bronze = spark.table(BRONZE_SALES_TABLE_PATH)

# 1. Type-cast
df_sales_typed = (
    df_sales_trx_bronze
    .select(
        col("TransactionID").cast(IntegerType()).alias("transaction_id"),
        col("CustomerID").cast(IntegerType()).alias("customer_id"),
        col("ProductID").cast(IntegerType()).alias("product_id"),
        col("QuantityPurchased").cast(IntegerType()).alias("quantity_purchased"),
        to_date(col("TransactionDate"), "dd/MM/yy").alias("transaction_date"),
        col("Price").cast(DoubleType()).alias("sales_price")
    )
)

# 2. check dataframe after type-cast

df_sales_typed.show(5)


+--------------+-----------+----------+------------------+----------------+-----------+
|transaction_id|customer_id|product_id|quantity_purchased|transaction_date|sales_price|
+--------------+-----------+----------+------------------+----------------+-----------+
|             1|        103|       120|                 3|      2023-01-01|      30.43|
|             2|        436|       126|                 1|      2023-01-01|      15.19|
|             3|        861|        55|                 3|      2023-01-01|      67.76|
|             4|        271|        27|                 2|      2023-01-01|      65.77|
|             5|        107|       118|                 1|      2023-01-01|      14.55|
+--------------+-----------+----------+------------------+----------------+-----------+
only showing top 5 rows


In [0]:
# 3. Dedeuplicate
# Create window per transaction_id ordered by transaction_date
window_dedup = Window.partitionBy("transaction_id").orderBy("transaction_date")

# Add row numbers
df_with_rownum = df_sales_typed.withColumn(
    "row_num", row_number().over(window_dedup)
)

# Keep earliest transaction
df_sales_deduped = (
    df_with_rownum
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# Rows that were removed as duplicates
df_removed_duplicates = (
    df_with_rownum
    .filter(col("row_num") > 1)
    .drop("row_num")
)

# Basic metrics
before_count = df_sales_typed.count()
after_count = df_sales_deduped.count()

print(f"Duplicates removed: {before_count - after_count}")
print(f"Rows after deduplication: {after_count}")

# Show removed duplicates
df_removed_duplicates.show()



Duplicates removed: 2
Rows after deduplication: 5000
+--------------+-----------+----------+------------------+----------------+-----------+
|transaction_id|customer_id|product_id|quantity_purchased|transaction_date|sales_price|
+--------------+-----------+----------+------------------+----------------+-----------+
|          4999|        904|       188|                 2|      2023-07-28|      36.22|
|          5000|        215|       159|                 2|      2023-07-28|      17.42|
+--------------+-----------+----------+------------------+----------------+-----------+



In [0]:
# 4: Handle nulls
df_sales_no_nulls = (
    df_sales_deduped
    .filter(
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &
        col("transaction_date").isNotNull()
    )
    # Impute null quantity with 1
    .withColumn(
        "quantity_purchased",
        when(col("quantity_purchased").isNull(), lit(1)).otherwise(col("quantity_purchased"))
    )
)

null_dropped = df_sales_deduped.count() - df_sales_no_nulls.count()
print(f"Rows dropped due to null critical fields: {null_dropped}")

# 5. Price correction — use inventory price as source of truth
df_products_price = (
    spark.table(f"{SILVER_SCHEMA_PATH}.product_inventory")
    .select(
        col("product_id"),
        col("price").alias("inventory_price")
    )
)

df_sales_corrected = (
    df_sales_no_nulls
    .join(df_products_price, on="product_id", how="left")
    .withColumn(
        "price_mismatch",
        when(col("sales_price") != col("inventory_price"), lit(True)).otherwise(lit(False))
    )
    # Use inventory price as corrected price
    .withColumn("price", col("inventory_price"))
    .withColumn("revenue", s_round(col("price") * col("quantity_purchased"), 2))
    .drop("sales_price", "inventory_price")
)

mismatches = df_sales_corrected.filter(col("price_mismatch")).count()
print(f"Price mismatches corrected: {mismatches}")
print(f"\nFinal Silver sales: {df_sales_corrected.count()} rows")
df_sales_corrected.show(5, truncate=False)

Rows dropped due to null critical fields: 0
Price mismatches corrected: 20

Final Silver sales: 5000 rows
+----------+--------------+-----------+------------------+----------------+--------------+-----+-------+
|product_id|transaction_id|customer_id|quantity_purchased|transaction_date|price_mismatch|price|revenue|
+----------+--------------+-----------+------------------+----------------+--------------+-----+-------+
|120       |1             |103        |3                 |2023-01-01      |false         |30.43|91.29  |
|126       |2             |436        |1                 |2023-01-01      |false         |15.19|15.19  |
|55        |3             |861        |3                 |2023-01-01      |false         |67.76|203.28 |
|27        |4             |271        |2                 |2023-01-01      |false         |65.77|131.54 |
|118       |5             |107        |1                 |2023-01-01      |false         |14.55|14.55  |
+----------+--------------+-----------+---------------

In [0]:
# 6. Write Silver sales transactions
(
    df_sales_corrected
    .withColumn("_processed_timestamp", current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA_PATH}.sales_transactions")
)
print(f"✓ {SILVER_SCHEMA_PATH}.sales_transactions written")

✓ retail_store.silver.sales_transactions written


In [0]:
# 7. check written tables
display(spark.table(f"{SILVER_SCHEMA_PATH}.sales_transactions").limit(5))

product_id,transaction_id,customer_id,quantity_purchased,transaction_date,price_mismatch,price,revenue,_processed_timestamp
120,1,103,3,2023-01-01,false,30.43,91.29,2026-03-16T10:34:02.082Z
126,2,436,1,2023-01-01,false,15.19,15.19,2026-03-16T10:34:02.082Z
55,3,861,3,2023-01-01,false,67.76,203.28,2026-03-16T10:34:02.082Z
27,4,271,2,2023-01-01,false,65.77,131.54,2026-03-16T10:34:02.082Z
118,5,107,1,2023-01-01,false,14.55,14.55,2026-03-16T10:34:02.082Z


# Customer Profiles

In [0]:
# Read Bronze customer profiles
df_customers_bronze = spark.table(f"{BRONZE_SCHEMA_PATH}.customer_profiles")

# Type-cast and clean
df_customers_silver = (
    df_customers_bronze
    .select(
        col("CustomerID").cast(IntegerType()).alias("customer_id"),
        col("Age").cast(IntegerType()).alias("age"),
        trim(col("Gender")).alias("gender"),
        trim(col("Location")).alias("location"),
        to_date(col("JoinDate"), "dd/MM/yy").alias("join_date")
    )
    .filter(col("customer_id").isNotNull())
    # Standardize nulls: empty strings -> null
    .withColumn("location", when(col("location") == "", lit(None)).otherwise(col("location")))
    .withColumn("gender", when(col("gender") == "", lit(None)).otherwise(col("gender")))
    .withColumn("_processed_timestamp", current_timestamp())
)

null_locations = df_customers_silver.filter(col("location").isNull()).count()
print(f"Customer profiles: {df_customers_silver.count()} rows")
print(f"Customers with null location: {null_locations}")
df_customers_silver.show(5, truncate=False)

# Write Silver customer profiles
(
    df_customers_silver
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.customer_profiles")
)
print(f"✓ {CATALOG}.{SILVER}.customer_profiles written")

Customer profiles: 1000 rows
Customers with null location: 13
+-----------+---+------+--------+----------+--------------------------+
|customer_id|age|gender|location|join_date |_processed_timestamp      |
+-----------+---+------+--------+----------+--------------------------+
|1          |63 |Other |East    |2020-01-01|2026-03-16 10:34:06.400126|
|2          |63 |Male  |North   |2020-01-02|2026-03-16 10:34:06.400126|
|3          |34 |Other |North   |2020-01-03|2026-03-16 10:34:06.400126|
|4          |19 |Other |NULL    |2020-01-04|2026-03-16 10:34:06.400126|
|5          |57 |Male  |North   |2020-01-05|2026-03-16 10:34:06.400126|
+-----------+---+------+--------+----------+--------------------------+
only showing top 5 rows
✓ retail_store.silver.customer_profiles written
